# One-region stimulus configurations


This notebook compares time-varying `external_input_ex_ex` signals with and without Gaussian noise. Each named stimulus uses the same nested configuration format as `01a` and `01b`.


## Initialization



Load environment variables

<font color=#fda> NOTE: </font> Prior to loading any modules, we need to load the environment variables from the configuration files. This way, we can configure the TVB modules using environment variables.


In [ ]:
from dotenv import dotenv_values, find_dotenv
import os
import sys
from pathlib import Path


# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }
os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))


Modules

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
from aux.environment import resolve_directory
from aux.tvb.stimulus import _build_pulse, _build_pulse_train
from aux.tvb.model import build_model
from aux.tvb.integrator import build_integrator
from aux.tvb.simulator import build_single_region_simulator
from aux.tvb.results import SimulationMetadata, ModelMetadata, ResultMetadata
from aux.tvb.plotting import plot_results


Console

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

Set main directory paths

In [ ]:

#ToDo: Cambiar el nombe de las variables de entorno
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Simulation values


In [ ]:
# 1) Simulation outputs.
OUTPUT_DTYPE = "float32"
OUTPUT_STATES = ["E"]

STATE_UNITS = {
    "E": "kHz", "I": "kHz",
    "C_ee": "kHz²", "C_ei": "kHz²", "C_ii": "kHz²",
    "W_e": "pA", "W_i": "pA", "ou_drift": "N.A.",
}
STIMULUS_UNITS = {"external_input_ex_ex": "kHz"}


In [ ]:
# 1.1) Simulation parameters (same values as 01b).
DURATION_MS = 2000.0
DT_MS = 0.5


In [ ]:
# 1.2) Stimulus configurations; each entry follows the 01a/01b format.
STIMULUS_CONFIGS = {
    "Pulse": {
        "external_input_ex_ex": {"pulse": {
            "baseline_khz": 0.005, "pulse_amplitude_khz": 0.04,
            "pulse_onset_ms": 500.0, "pulse_duration_ms": 800.0,
        }},
    },
    "Periodic": {
        "external_input_ex_ex": {"pulse_train": {
            "baseline_khz": 0.005, "pulse_amplitude_khz": 0.04,
            "pulse_onset_ms": 500.0, "pulse_duration_ms": 120.0,
            "pulse_period_ms": 250.0,
        }},
    },
    "Vectors": {
        "external_input_ex_ex": {"vectors": {
            "baseline_khz": 0.005, "onset_ms": 500.0,
            "points_ms": [0.0, 250.0, 500.0, 850.0],
            "amplitudes_khz": [0.01, 0.04, 0.015, 0.03],
        }},
    },
    "Random intervals": {
        "external_input_ex_ex": {"random_intervals": {
            "baseline_khz": 0.005, "onset_ms": 500.0,
            "min_duration_ms": 100.0, "max_duration_ms": 250.0,
            "min_amplitude_khz": 0.01, "max_amplitude_khz": 0.04,
            "seed": 7,
        }},
    },
    "Random interpolation": {
        "external_input_ex_ex": {"random_interpolation": {
            "baseline_khz": 0.005, "onset_ms": 500.0,
            "min_duration_ms": 100.0, "max_duration_ms": 250.0,
            "min_amplitude_khz": 0.01, "max_amplitude_khz": 0.04,
            "seed": 7,
        }},
    },
}

NOISE_CONFIG = {"mean_khz": 0.0, "std_khz": 0.002, "noise_seed": 7}

NOISE_STIMULUS_CONFIG = {
    name: {
        parameter: {
            func: {**values, **NOISE_CONFIG}
            for func, values in definition.items()
            if func in ("pulse", "pulse_train", "vectors", "random_intervals", "random_interpolation")
        }
        for parameter, definition in stimulus.items() 
        if definition is not None 
    }
    for name, stimulus in STIMULUS_CONFIGS.items()
}

pprint('Noise stimulus configuration', NOISE_STIMULUS_CONFIG)

In [ ]:
# 1.3) Parámetros del modelo Zerlaut-AdEx.
#MODEL_NAME = "ZerlautAdaptationFirstOrder"
#MODEL_NAME = "ZerlautAdaptationSecondOrder"


MODEL_PARAMETERS = {
    # Población y conectividad
    "N_tot": 10000,
    "p_connect_e": 0.05,
    "p_connect_i": 0.05,
    "g": 0.2,
    "K_ext_e": 400,
    "K_ext_i": 35,
    
    # Parametros de membrana
    "g_L": 10.0,
    "C_m": 200.0,
    "E_L_e": -60.0,
    "E_L_i": -65.0,

    # Parametors de adaptación
    "a_e": 4.0,
    "b_e": 60.0,
    "tau_w_e": 200.0,
    "a_i": 0.0,
    "b_i": 20.0,
    "tau_w_i": 100.0,

    # Parametros sinápticos
    "E_e": 0.0,
    "E_i": -80.0,
    "Q_e": 1.0,
    "Q_i": 5.0,
    "tau_e": 5.0,
    "tau_i": 5.0,
    
    "T": 10.0,

    # Parametro de segundo orden (no hay un motivo matemático)
    "S_i": 1, # Escala la entrada remota recibida por la población inhibidora en el modelo de segundo orden.

    # Parametros de ruido
    "weight_noise": 2e-4, #10.5
    "tau_OU": 2, #0.5
}

# Todos los parámetros son de tipo float menos lo que están en esta lista. 
INTEGER_PARAMETERS = {"N_tot", "K_ext_e", "K_ext_i"}


## Build the stimulus


Each named configuration describes one simulation, with the same parameter-to-stimulus mapping as `01a`. `build_stimulus` generates its direct signals first, then scales signals from parameters already defined in that simulation. Gaussian noise is added by a separate function using its mean and standard deviation; a standard deviation of zero leaves the signal unchanged. Vector points are relative to onset and are linearly interpolated.


In [ ]:
def _step_stimulation(time_ms, baseline_khz, onset_ms, points_ms, amplitudes_khz):
    points = np.asarray(points_ms, dtype=float)
    amplitudes = np.asarray(amplitudes_khz, dtype=float)

    if (points.ndim != 1 or amplitudes.ndim != 1 or len(points) == 0
            or len(points) != len(amplitudes) or points[0] < 0
            or not np.all(np.isfinite(points)) or not np.all(np.isfinite(amplitudes))
            or np.any(np.diff(points) <= 0)):
        raise ValueError("points_ms and amplitudes_khz need matching, ordered finite values")
    if points[0] > 0:
        points = np.r_[0.0, points]
        amplitudes = np.r_[baseline_khz, amplitudes]
    signal = np.full(time_ms.shape, baseline_khz, dtype=float)
    active = time_ms >= onset_ms
    signal[active] = np.interp(time_ms[active] - onset_ms, points, amplitudes)
    return signal


def _random_stimulation(time_ms, baseline_khz, onset_ms, min_duration_ms,
                        max_duration_ms, min_amplitude_khz, max_amplitude_khz,
                        seed=None, interpolate=False):
    if (min_duration_ms <= 0 or max_duration_ms < min_duration_ms
            or max_amplitude_khz < min_amplitude_khz):
        raise ValueError("Invalid duration or amplitude range")
    rng = np.random.default_rng(seed)
    signal = np.full(time_ms.shape, baseline_khz, dtype=float)
    start_ms = float(onset_ms)
    previous_amplitude = baseline_khz
    while start_ms <= time_ms[-1]:
        duration_ms = rng.uniform(min_duration_ms, max_duration_ms)
        next_amplitude = rng.uniform(min_amplitude_khz, max_amplitude_khz)
        interval = (time_ms >= start_ms) & (time_ms < start_ms + duration_ms)
        if interpolate:
            fraction = (time_ms[interval] - start_ms) / duration_ms
            signal[interval] = previous_amplitude + fraction * (next_amplitude - previous_amplitude)
        else:
            signal[interval] = next_amplitude
        previous_amplitude = next_amplitude
        start_ms += duration_ms
    return signal


def _random_step_stimulation(time_ms, baseline_khz, onset_ms, min_duration_ms,
                                max_duration_ms, min_amplitude_khz, max_amplitude_khz,
                                seed=None):
    return _random_stimulation(time_ms, baseline_khz, onset_ms, min_duration_ms,
                               max_duration_ms, min_amplitude_khz, max_amplitude_khz,
                               seed, False)


def _random_interpolated_stimulation(time_ms, baseline_khz, onset_ms, min_duration_ms,
                                    max_duration_ms, min_amplitude_khz, max_amplitude_khz,
                                    seed=None):
    return _random_stimulation(time_ms, baseline_khz, onset_ms, min_duration_ms,
                               max_duration_ms, min_amplitude_khz, max_amplitude_khz,
                               seed, True)


STIMULUS_BUILDERS = {
    "pulse": _build_pulse,
    "pulse_train": _build_pulse_train,
    "vectors": _step_stimulation,
    "random_intervals": _random_step_stimulation,
    "random_interpolation": _random_interpolated_stimulation,
}

NOISE_FIELDS = {"mean_khz", "std_khz", "noise_seed"}


def _add_gaussian_noise(signal, std_khz=0, mean_khz=0, seed=None):
    if std_khz == 0: return signal
    
    rng = np.random.default_rng(seed)
    return signal + rng.normal(mean_khz, std_khz, signal.shape)


def build_stimulus(stimulus, duration_ms, dt_ms):
    time_ms = np.arange(0.0, duration_ms, dt_ms)
    output = {}
    definitions = {}

    for key, value in stimulus.items():
        if value is None:  continue
        
        if not isinstance(value, dict):
            raise ValueError(f"Stimulus value for {key} must be a dictionary")
        
        # Se quita los parametros de ruido.
        definition = {name: parameters for name, parameters in value.items()
                      if name not in NOISE_FIELDS}

        if len(definition) != 1:
            raise ValueError(f"Stimulus value for {key} must have one type or source")
        
        kind, parameters = next(iter(definition.items()))
        definitions[key] = (kind, parameters)
        
        if kind in STIMULUS_BUILDERS:
            signal = STIMULUS_BUILDERS[kind](time_ms, **parameters)
            output[key] = _add_gaussian_noise(signal, value)

    for key, (source, scale) in definitions.items():
        if source in stimulus:
            if source not in output:
                raise ValueError(f"Stimulus value for {key} requires '{source}' to be defined")
            output[key] = _add_gaussian_noise(scale * output[source], stimulus[key], noise_seed)
        elif source not in STIMULUS_BUILDERS:
            raise ValueError(f"Unknown stimulus type or source: {source}")

    return time_ms, [(key, output[key][:, np.newaxis]) for key in output]


In [ ]:
stimuli = {}
for condition, configurations in (("Without noise", STIMULUS_CONFIGS),
                                  ("With Gaussian noise", NOISE_STIMULUS_CONFIG)):
    stimuli[condition] = {}
    for name, stimulus_config in configurations.items():
        time_ms, stimulus = build_stimulus(stimulus_config, DURATION_MS, DT_MS, noise_seed=NOISE_SEED)
        stimuli[condition][name] = stimulus
pprint('Stimulus configurations:', list(STIMULUS_CONFIGS))


## Deterministic Simulation


### Run simulation


In [ ]:
integrator = build_integrator(DT_MS, integrator_class="HeunDeterministic")
results = {}
for condition, configurations in stimuli.items():
    results[condition] = {}
    for name, stimulus in configurations.items():
        model = build_model(
            MODEL_PARAMETERS, INTEGER_PARAMETERS,
            model_class="ZerlautAdaptationFirstOrder",
            variable_of_interest=OUTPUT_STATES,
        )
        simulator = build_single_region_simulator(model, stimulus, integrator)
        time_sim, raw = simulator.run(n_steps=len(time_ms))[0]
        results[condition][name] = (
            time_sim, {"E": raw[:, model.state_variables.index("E"), 0, 0]}
        )
        pprint(f"{condition}, {name}: {len(time_sim)} samples")


### Plot results


In [ ]:
metadata = ResultMetadata(
    simulation=SimulationMetadata(DURATION_MS, DT_MS, integrator, OUTPUT_DTYPE),
    model=ModelMetadata(
        name="ZerlautAdaptationFirstOrder", parameters=MODEL_PARAMETERS,
        integer_parameters=list(INTEGER_PARAMETERS), output_variables=OUTPUT_STATES,
        state_units=STATE_UNITS, stimulus_units=STIMULUS_UNITS,
    ),
    stimulus={},
)
for condition, condition_results in results.items():
    fig = plt.figure(figsize=(4.5 * len(condition_results), 6), constrained_layout=True)
    grid = fig.add_gridspec(2, len(condition_results))
    fig.suptitle(condition)
    for column, (name, (time_sim, states)) in enumerate(condition_results.items()):
        stimulus_ax = fig.add_subplot(grid[0, column])
        for parameter, signal in stimuli[condition][name]:
            stimulus_ax.plot(time_ms, signal[:, 0], label=parameter)
        stimulus_ax.set_title(name)
        stimulus_ax.set_ylabel("Stimulus (kHz)")
        stimulus_ax.grid(alpha=0.25)
        stimulus_ax.legend()
        plot_results(time_sim, states, metadata, show_stimulus=False,
                     ax=fig.add_subplot(grid[1, column]), plot_states=["E"])
    plt.show()
